In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from mlxtend.frequent_patterns import apriori, association_rules

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "DejaVu Serif"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [11]:
BASE_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
DADOS_BRASIL = BASE_DIR / "dados" / "brasil"

ARQ_NASCIMENTOS = DADOS_BRASIL / "ano_nascimento.csv"
ARQ_IDADE = DADOS_BRASIL / "idade_mae.csv"
ARQ_INSTRUCAO = DADOS_BRASIL / "instruçao_mae.csv"
ARQ_PRE_NATAL = DADOS_BRASIL / "pre-natal.csv"
ARQ_PARTO = DADOS_BRASIL / "tipo_parto.csv"

In [12]:
def ler_datasus(caminho):
    df = pd.read_csv(caminho, sep=";", encoding="latin1", skiprows=3, dtype=str)
    df = df.dropna(how="all")
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    return df

def limpar_nome_uf(valor):
    return str(valor).replace("..", "").strip()

def converter_numero(serie):
    return pd.to_numeric(
        serie.astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .replace("-", np.nan),
        errors="coerce"
    )

def preparar_datasus_por_uf(caminho):
    df = ler_datasus(caminho)
    col_uf = "Região/Unidade da Federação"
    df[col_uf] = df[col_uf].apply(limpar_nome_uf)

    excluir = [
        "Região Norte", "Região Nordeste", "Região Sudeste",
        "Região Sul", "Região Centro-Oeste", "Total", "nan"
    ]

    df = df[~df[col_uf].isin(excluir)].copy()
    df = df.rename(columns={col_uf: "UF"})

    for col in df.columns:
        if col != "UF":
            df[col] = converter_numero(df[col])

    return df

def proporcao(df, numerador, denominador="Total"):
    return df[numerador] / df[denominador] * 100

In [13]:
nasc = preparar_datasus_por_uf(ARQ_NASCIMENTOS)
anos = [str(ano) for ano in range(2000, 2025)]
nasc = nasc[["UF"] + anos].copy()

base = nasc[["UF", "2000", "2024"]].copy()
base["Queda_percentual"] = ((base["2000"] - base["2024"]) / base["2000"]) * 100

idade = preparar_datasus_por_uf(ARQ_IDADE)
idade["Mae_30_34"] = proporcao(idade, "30 a 34 anos")
idade["Mae_35_39"] = proporcao(idade, "35 a 39 anos")
idade = idade[["UF", "Mae_30_34", "Mae_35_39"]]

instrucao = preparar_datasus_por_uf(ARQ_INSTRUCAO)
instrucao["Instrucao_12_mais"] = proporcao(instrucao, "12 anos e mais")
instrucao = instrucao[["UF", "Instrucao_12_mais"]]

prenatal = preparar_datasus_por_uf(ARQ_PRE_NATAL)
prenatal["Pre_natal_7_mais"] = proporcao(prenatal, "7 ou mais consultas")
prenatal = prenatal[["UF", "Pre_natal_7_mais"]]

parto = preparar_datasus_por_uf(ARQ_PARTO)
parto["Cesarea"] = proporcao(parto, "Cesário")
parto = parto[["UF", "Cesarea"]]

base_assoc = (
    base
    .merge(idade, on="UF", how="inner")
    .merge(instrucao, on="UF", how="inner")
    .merge(prenatal, on="UF", how="inner")
    .merge(parto, on="UF", how="inner")
)

base_assoc.head()

,UF,2000,2024,Queda_percentual,Mae_30_34,Mae_35_39,Instrucao_12_mais,Pre_natal_7_mais,Cesarea
0,Rondônia,31307.0,21716.0,30.635321,14.376707,6.290258,14.170719,51.109390,60.638354
1,Acre,15228.0,13101.0,13.967691,13.848289,7.154202,11.378238,36.153322,35.311389
2,Amazonas,67646.0,65950.0,2.507170,13.562784,6.839980,9.811204,38.276261,35.137902
3,Roraima,9744.0,12090.0,-24.076355,14.262397,7.148313,14.181326,40.871749,32.109579
4,Pará,126340.0,118566.0,6.153237,12.200850,5.600722,9.883527,39.818082,42.649223


In [14]:
variaveis = [
    "Queda_percentual",
    "Mae_30_34",
    "Mae_35_39",
    "Instrucao_12_mais",
    "Pre_natal_7_mais",
    "Cesarea"
]

nomes_binarios = {
    "Queda_percentual": "Queda_nascimento_alta",
    "Mae_30_34": "Mae_30_34_alta",
    "Mae_35_39": "Mae_35_39_alta",
    "Instrucao_12_mais": "Instrucao_12_mais_alta",
    "Pre_natal_7_mais": "Pre_natal_7_mais_alto",
    "Cesarea": "Cesarea_alta"
}

base_binaria = pd.DataFrame()
base_binaria["UF"] = base_assoc["UF"]

for var in variaveis:
    mediana = base_assoc[var].median()
    base_binaria[nomes_binarios[var]] = base_assoc[var] >= mediana
    print(f"{var}: mediana = {mediana:.2f}")

base_binaria.head()

Queda_percentual: mediana = 26.46
Mae_30_34: mediana = 16.50
Mae_35_39: mediana = 8.10
Instrucao_12_mais: mediana = 14.18
Pre_natal_7_mais: mediana = 55.44
Cesarea: mediana = 49.63


,UF,Queda_nascimento_alta,Mae_30_34_alta,Mae_35_39_alta,Instrucao_12_mais_alta,Pre_natal_7_mais_alto,Cesarea_alta
0,Rondônia,True,False,False,False,False,True
1,Acre,False,False,False,False,False,False
2,Amazonas,False,False,False,False,False,False
3,Roraima,False,False,False,True,False,False
4,Pará,False,False,False,False,False,False


In [15]:
transacoes = base_binaria.drop(columns="UF")

itemsets = apriori(
    transacoes,
    min_support=0.25,
    use_colnames=True
)

regras = association_rules(
    itemsets,
    metric="confidence",
    min_threshold=0.60
)

regras["antecedents"] = regras["antecedents"].apply(lambda x: ", ".join(list(x)))
regras["consequents"] = regras["consequents"].apply(lambda x: ", ".join(list(x)))

regras = regras.sort_values(
    by=["lift", "confidence", "support"],
    ascending=False
)

regras[["antecedents", "consequents", "support", "confidence", "lift"]].head(20)

,antecedents,consequents,support,confidence,lift
63,"Pre_natal_7_mais_alto, Cesarea_alta","Instrucao_12_mais_alta, Mae_30_34_alta",0.270270,0.833333,3.083333
62,"Instrucao_12_mais_alta, Mae_30_34_alta","Pre_natal_7_mais_alto, Cesarea_alta",0.270270,1.000000,3.083333
64,"Pre_natal_7_mais_alto, Mae_30_34_alta","Instrucao_12_mais_alta, Cesarea_alta",0.270270,0.833333,2.803030
65,"Mae_30_34_alta, Cesarea_alta","Instrucao_12_mais_alta, Pre_natal_7_mais_alto",0.270270,0.833333,2.803030
60,"Instrucao_12_mais_alta, Pre_natal_7_mais_alto","Mae_30_34_alta, Cesarea_alta",0.270270,0.909091,2.803030
61,"Instrucao_12_mais_alta, Cesarea_alta","Pre_natal_7_mais_alto, Mae_30_34_alta",0.270270,0.909091,2.803030
50,"Instrucao_12_mais_alta, Pre_natal_7_mais_alto",Cesarea_alta,0.297297,1.000000,2.642857
51,"Instrucao_12_mais_alta, Cesarea_alta",Pre_natal_7_mais_alto,0.297297,1.000000,2.642857
28,"Mae_35_39_alta, Cesarea_alta",Mae_30_34_alta,0.270270,1.000000,2.642857
33,"Instrucao_12_mais_alta, Mae_30_34_alta",Pre_natal_7_mais_alto,0.270270,1.000000,2.642857


In [16]:
regras_queda = regras[
    regras["consequents"].str.contains("Queda_nascimento_alta") |
    regras["antecedents"].str.contains("Queda_nascimento_alta")
].copy()

regras_queda = regras_queda.sort_values(
    by=["lift", "confidence", "support"],
    ascending=False
)

regras_queda[["antecedents", "consequents", "support", "confidence", "lift"]].head(15)

,antecedents,consequents,support,confidence,lift
0,Mae_35_39_alta,Queda_nascimento_alta,0.297297,0.785714,2.076531
1,Queda_nascimento_alta,Mae_35_39_alta,0.297297,0.785714,2.076531


In [17]:
tabela_final = regras.sort_values(
    by=["lift", "confidence", "support"],
    ascending=False
).head(8).copy()

tabela_final = tabela_final[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]]

tabela_final = tabela_final.rename(columns={
    "antecedents": "Antecedente",
    "consequents": "Consequente",
    "support": "Suporte",
    "confidence": "Confiança",
    "lift": "Lift"
})

for col in ["Suporte", "Confiança", "Lift"]:
    tabela_final[col] = tabela_final[col].round(3)

tabela_final

,Antecedente,Consequente,Suporte,Confiança,Lift
63,"Pre_natal_7_mais_alto, Cesarea_alta","Instrucao_12_mais_alta, Mae_30_34_alta",0.270,0.833,3.083
62,"Instrucao_12_mais_alta, Mae_30_34_alta","Pre_natal_7_mais_alto, Cesarea_alta",0.270,1.000,3.083
64,"Pre_natal_7_mais_alto, Mae_30_34_alta","Instrucao_12_mais_alta, Cesarea_alta",0.270,0.833,2.803
65,"Mae_30_34_alta, Cesarea_alta","Instrucao_12_mais_alta, Pre_natal_7_mais_alto",0.270,0.833,2.803
60,"Instrucao_12_mais_alta, Pre_natal_7_mais_alto","Mae_30_34_alta, Cesarea_alta",0.270,0.909,2.803
61,"Instrucao_12_mais_alta, Cesarea_alta","Pre_natal_7_mais_alto, Mae_30_34_alta",0.270,0.909,2.803
50,"Instrucao_12_mais_alta, Pre_natal_7_mais_alto",Cesarea_alta,0.297,1.000,2.643
51,"Instrucao_12_mais_alta, Cesarea_alta",Pre_natal_7_mais_alto,0.297,1.000,2.643
